# 1. Ma’lumotlarni tayyorlash va tahlil qilish

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [2]:
!wget https://github.com/Ankit152/IMDB-sentiment-analysis/raw/master/IMDB-Dataset.csv

--2025-10-17 12:39:52--  https://github.com/Ankit152/IMDB-sentiment-analysis/raw/master/IMDB-Dataset.csv
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv [following]
--2025-10-17 12:39:53--  https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [text/plain]
Saving to: ‘IMDB-Dataset.csv.1’

IMDB-Dataset.csv.1  100%[===================>]  63.14M   202MB/s    in 0.3s    

2025-10-17 12:39:55 (202 MB/s) - ‘IMDB-Dataset.csv.1’ saved [66212309/6621

In [3]:
df = pd.read_csv('IMDB-Dataset.csv')
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [5]:
# Har bir toifadan (positive va negative) 1000 tadan tasodifiy namunani ajratib oling
# (random_state=101 bilan)

df2 = (df.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(n=1000, random_state=101))
      .sample(frac=1, random_state=101)
      .reset_index(drop=True)
)

/tmp/ipython-input-2661563647.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df2 = (df.groupby("sentiment", group_keys=False).apply(lambda x: x.sample(n=1000, random_state=101))


In [6]:
df2.shape

(2000, 2)

# 2. RNN modeli uchun ma’lumotlarni kodlash

### 2.1. Lug‘at (vocabulary) yaratish

In [7]:
# 1. Barcha sharhlardagi so‘zlarni tokenize qiling.
def tokenize(text):
  return text.lower().split()

In [8]:
# 2. collections.Counter yordamida so‘zlar chastotasini hisoblang.
counter = Counter(word for text in df2['review'] for word in tokenize(text))

In [9]:
# 3. Eng ko‘p uchraydigan 4000 ta so‘zdan iborat lug‘at (vocab) yarating.
most_common = counter.most_common(4000)

In [10]:
# 4. Lug‘atga <pad> (indeks 0) va <unk> (indeks 1) maxsus tokenlarini qo‘shing.
vocab = {word: i+2 for i, (word, _) in enumerate(most_common)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

### 2.2. Matnni sonli ketma-ketlikka o‘girish

In [11]:
def encode(text):
  return torch.tensor([vocab.get(word, 1) for word in tokenize(text)])

# 3. SentimentRNN modelini qurish va o‘qitish

### 3.1. Model arxitekturasini yaratish

  SentimentRNN nomli Pytorch nn.Module klassini yarating. Arxitektura quyidagicha bo‘lsin:

    - embedding_dim: 128
    - hidden_dim: 256
    - nn.Embedding
    - nn.RNN
    – nn.Linear
    - nn.Sigmoid



In [12]:
class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256):
    super(SentimentRNN, self).__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)
    _, h = self.rnn(x)
    x = self.fc(h.squeeze(0))
    return self.sigmoid(x)

##3.2. Modelni sozlash va o‘qitish

In [13]:
# 1 Modelni, nn.BCELoss yo‘qotish funksiyasini va Adam optimizatorini lr=0.001 bilan sozlang.
model_rnn = SentimentRNN(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=0.001)

In [14]:
# 2 DataLoader yarating.

class SimpleDataset(nn.Module):
  def __init__(self, df):
    self.X = [encode(text) for text in df['review']]
    self.y = [torch.tensor([1.0 if s == 'positive' else 0.0]) for s in df['sentiment']]

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [15]:
def collate_fn(batch):
  Xs, ys = zip(*batch)
  max_len = max(len(x) for x in Xs)
  padded_X = [torch.cat([x, torch.zeros(max_len - len(x))]) for x in Xs]
  return torch.stack(padded_X).long(), torch.stack(ys)

train_loader = DataLoader(SimpleDataset(df2), batch_size=32, shuffle=True, collate_fn=collate_fn)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model_rnn.to(device)

cuda


In [17]:
# 3 Modelni 10 ta epoxa davomida o‘rgating va har bir epoxadagi o‘rtacha yo‘qotishni ekranga chiqaring.

epochs_list = []
loss_list = []

for epoch in range(10):
  total_loss = 0

  for X, y in train_loader:
    X, y = X.to(device), y.to(device)

    y_pred = model_rnn(X)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(train_loader)
  epochs_list.append(epoch + 1)
  loss_list.append(avg_loss)

  print(f'Epoch: {epoch+1}, Loss: {avg_loss:.4f}')

Epoch: 1, Loss: 0.7066
Epoch: 2, Loss: 0.7048
Epoch: 3, Loss: 0.6996
Epoch: 4, Loss: 0.6911
Epoch: 5, Loss: 0.7070
Epoch: 6, Loss: 0.7063
Epoch: 7, Loss: 0.6979
Epoch: 8, Loss: 0.6989
Epoch: 9, Loss: 0.7012
Epoch: 10, Loss: 0.6996


In [18]:
# evaluation
def predict_rnn(text, model):
  model.eval()
  model.to(device)
  with torch.no_grad():
    x = encode(text)
    x = x.unsqueeze(0).to(device)
    y_pred = model(x)

    prob = y_pred.item()
    label = 'positive' if prob > 0.5 else 'negative' # bashorat

  print(f"Probability: {prob:.2f} | Label: {label}")

In [19]:
predict_rnn("Taomlari juda mazali, lekin xizmat ko‘rsatish juda sekin.", model_rnn)
predict_rnn("Bu restoranga boshqa qaytib kelmayman. Ofitsiant juda qo‘pol edi.", model_rnn)

Probability: 0.52 | Label: positive
Probability: 0.57 | Label: positive


# 4. Transformer modeli uchun ma’lumotlarni tayyorlash

In [20]:
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

In [21]:
!pip install -U datasets fsspec transformers

### 4.1. Ma’lumotlar to‘plamini yuklash

datasets kutubxonasi yordamida 2-Milestone'da tayyorlangan final_df dan Dataset obyekti yarating.<br>Ma’lumotlarni train (2500 ta) va test (500 ta) qismlariga ajrating (seed=123).

In [22]:
def encode_labels(example):
    if "sentiment" in example:
        example["labels"] = 1 if example["sentiment"] == "positive" else 0
    return example

In [23]:
dataset = Dataset.from_pandas(df2)
split_dataset = dataset.train_test_split(test_size=500, seed=123)

train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

In [24]:
train_dataset = train_dataset.map(encode_labels)
test_dataset = test_dataset.map(encode_labels)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### 4.2. Matnni tokenizatsiya qilish

In [25]:
# 1. distilbert-base-uncased tokenizer’ini yuklang.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [26]:
# 2. Matnlarni tokenizatsiya qiluvchi, padding va truncation qo‘llaydigan preprocess_function yarating (max_length=256).
def preprocess_function(examples):
    return tokenizer(examples["review"], truncation=True, padding='max_length', max_length=256)


In [27]:
# 3. Funksiyani train va test to‘plamlariga map metodi orqali qo‘llang.
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [28]:
tokenized_train = tokenized_train.remove_columns(["review", "sentiment"])
tokenized_test = tokenized_test.remove_columns(["review", "sentiment"])

In [29]:
print(tokenized_train.column_names)
print(tokenized_test.column_names)

['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']


In [30]:
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## 5. Transformer modelini o‘qitish va baholash

### 5.1. Modelni yuklash va sozlash

In [31]:
# 1. distilbert-base-uncased asosida AutoModelForSequenceClassification modelini yuklang
model_transformer = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
# 2. TrainingArguments sozlamalarini yarating: num_train_epochs=3, logging_steps=100.
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    logging_dir="./logs",
    logging_steps=100,
    report_to='none'
)

# 3. Trainer obyektini yarating.
trainer = Trainer(
    model=model_transformer,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
  )

### 5.2. Modelni o‘qitish va baholash

In [33]:
# 1. trainer.train() metodi bilan modelni shug‘ullantiring.
trainer.train()

Step,Training Loss
100,0.343600


TrainOutput(global_step=141, training_loss=0.27706953143397123, metrics={'train_runtime': 107.8477, 'train_samples_per_second': 41.726, 'train_steps_per_second': 1.307, 'total_flos': 298051646976000.0, 'train_loss': 0.27706953143397123, 'epoch': 3.0})

In [34]:
# 2. trainer.evaluate() metodini chaqirib, modelning eval_loss ko‘rsatkichini tahlil qiling
trainer.evaluate()

{'eval_loss': 0.3711109757423401,
 'eval_runtime': 3.4381,
 'eval_samples_per_second': 145.429,
 'eval_steps_per_second': 4.654,
 'epoch': 3.0}

# 6. Ikkala modelni taqqoslash va xulosa

## 6.1. Bashorat funksiyalarini yaratish

### Har bir model (SentimentRNN va Transformer) uchun yangi matnli sharhni qabul qilib, uning toifasini (“positive” yoki “negative”) bashorat qiluvchi alohida funksiyalar yarating.

In [35]:
def predict_rnn(texts, model):
  model.eval()
  model.to(device)
  with torch.no_grad():
    for text in texts:
      x = encode(text)
      x = x.unsqueeze(0).to(device)
      y_pred = model(x)

      prob = y_pred.item()
      label = 'positive' if prob > 0.5 else 'negative'

      print(f"[SentimentRNN]: Text: {text} | Probability: {prob:.2f} | Label: {label}")

In [36]:
def predict_transformer(texts, model):
  model.eval()
  inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128).to('cuda')

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

  label_map = {0: 'negative', 1: 'positive'}

  for text, pred, prob in zip(texts, predictions, probs):
    print(f"[Transformer]: Text: {text} | Probability: {prob[pred.item()].item()} | Label: {label_map[pred.item()]}")

## 6.2. Modellarni sinovdan o‘tkazish va taqqoslash
Quyidagi yangi sharhlar uchun har ikkala modelning bashoratlarini oling va natijalarni chop eting:

a) Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.

b) Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.

c) Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.

In [37]:
texts = [
    "Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi",
    "Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan",
    "Filmning vizual effektlari yaxshi, ammo hikoyasi zaif."
]


In [38]:
predict_rnn(texts, model=model_rnn)

[SentimentRNN]: Text: Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi | Probability: 0.52 | Label: positive
[SentimentRNN]: Text: Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan | Probability: 0.52 | Label: positive
[SentimentRNN]: Text: Filmning vizual effektlari yaxshi, ammo hikoyasi zaif. | Probability: 0.57 | Label: positive


In [39]:
predict_transformer(texts, model=model_transformer)

[Transformer]: Text: Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi | Probability: 0.5364159345626831 | Label: negative
[Transformer]: Text: Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan | Probability: 0.5853957533836365 | Label: positive
[Transformer]: Text: Filmning vizual effektlari yaxshi, ammo hikoyasi zaif. | Probability: 0.6678231954574585 | Label: positive


## 6.3. Xulosa yozish
Olingan natijalar (o‘qitishdagi yo‘qotish ko‘rsatkichlari, baholash natijalari va yangi sharhlardagi bashoratlar) asosida qaysi model bu vazifa uchun yaxshiroq ekanligi haqida 2-3 jumlalik xulosa yozing.

In [39]:
"""
  Loss ga ko'ra rnn model stabil ravishda tushmaga. Goh tushib, goh ko'tarilgan. Tushishi sekin.
  Predictionlari ham notog'ri.

  Transformer loss 100 epoxada 0.34 ni ko'rsatyapti. Predictionlari deyarli to'g'ri chiqdi.

  ikkala modelning ham giperparametrlarini o'zgartirib, yaxshi natijaga erishguncha train qilish kerak deb o'ylayman.

"""